# KV Cache Playground

This notebook uses a tiny local transformer to inspect KV-cache shapes and memory. The model is randomly initialized, which is fine for cache inspection: KV-cache size depends on architecture, batch size, sequence length, and dtype.

In [ ]:
import torch
from kv_cache_playground import TinyConfig, TinyTransformer, cache_bytes, mib, theoretical_cache_bytes

torch.manual_seed(0)

## Change These Parameters

Try changing `n_layers`, `n_heads`, `n_kv_heads`, `batch_size`, `prompt_len`, `new_tokens`, and `dtype`. For grouped-query attention, set `n_kv_heads` smaller than `n_heads`; for multi-query attention, set `n_kv_heads = 1`.

In [ ]:
batch_size = 1
prompt_len = 8
new_tokens = 6

cfg = TinyConfig(
    vocab_size=256,
    d_model=128,
    n_layers=2,
    n_heads=4,
    n_kv_heads=4,
)

dtype = torch.float32
device = "cpu"
cfg.validate()
model = TinyTransformer(cfg).to(device=device, dtype=dtype).eval()

print(cfg)
print("head_dim:", cfg.head_dim)

## Run Prompt Prefill

During prefill, the model receives the whole prompt. Each layer stores key and value tensors shaped `(batch, n_kv_heads, cached_tokens, head_dim)`.

In [ ]:
@torch.no_grad()
def show_cache(label, caches):
    key, value = caches[0]
    print(label)
    print("layer 0 key shape:  ", tuple(key.shape))
    print("layer 0 value shape:", tuple(value.shape))
    print("all layers:", len(caches))
    print("actual cache memory:", f"{mib(cache_bytes(caches)):.4f} MiB")
    print()

input_ids = torch.randint(0, cfg.vocab_size, (batch_size, prompt_len), device=device)
with torch.no_grad():
    logits, caches = model(input_ids)

show_cache(f"After prompt prefill ({prompt_len} tokens)", caches)

## Decode One Token At A Time

During decoding, only the newest token is passed through the model, but the cached keys and values keep growing by one token per step.

In [ ]:
with torch.no_grad():
    for step in range(1, new_tokens + 1):
        next_token = logits[:, -1:].argmax(dim=-1)
        logits, caches = model(next_token, caches)
        cached_tokens = prompt_len + step
        expected = theoretical_cache_bytes(
            batch_size=batch_size,
            seq_len=cached_tokens,
            n_layers=cfg.n_layers,
            n_kv_heads=cfg.n_kv_heads,
            head_dim=cfg.head_dim,
            dtype=dtype,
        )
        show_cache(f"After decode step {step} ({cached_tokens} cached tokens)", caches)
        print("formula memory:", f"{mib(expected):.4f} MiB")
        print("formula: batch * tokens * layers * 2(K,V) * n_kv_heads * head_dim * dtype_bytes")
        print("-" * 72)

## Things To Try

- Increase `prompt_len` or `new_tokens`: cache memory grows linearly with total cached tokens.
- Increase `n_layers`: cache memory grows linearly with layers.
- Set `n_kv_heads = 1`: this mimics multi-query attention and greatly reduces cache memory.
- Use `torch.float16` or `torch.bfloat16`: cache memory roughly halves compared with `torch.float32`.